# Web Crawling

**Seong Cho** · ACME Data Science Essentials

**Lab objective:** gather data spread over several web pages by crawling between them without straining servers or violating terms of use, and use Selenium for pages that load content asynchronously or need user interaction.

<sub>Lab specification from the January 2019 edition of the *Data Science Essentials* lab manual (where it is Lab 6; it is not in the August 2018 edition) of [Foundations of Applied Mathematics](https://foundations-of-applied-mathematics.github.io/) (BYU ACME), © Dr. J. Humpherys, used under [CC BY 3.0 US](http://creativecommons.org/licenses/by/3.0/us/). Problem statements are quoted from my lab template and the spec; the notes are condensed from the spec. Solutions are my own.</sub>

> **Note.** This notebook was not re-run for this archive: it needs live access to websites as they were in late 2018 (several have since changed) and Selenium with a Chrome driver.

In [ ]:
# """Volume 3: Web Scraping 2.
# <Name>
# <Class>
# <Date>
# """
import re
import time
import requests
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.common.exceptions import NoSuchElementException
import numpy as np
import matplotlib.pyplot as plt

## Scraping etiquette

Scraping becomes a problem for a site owner when it ignores the site's terms and conditions (or gathers private or proprietary data), or when it loads the server with too many requests in quick succession. Sites that allow scraping usually publish a `robots.txt` file (e.g. www.google.com/robots.txt) saying which parts are off-limits and how often requests may be made (the robots exclusion standard).

> **Achtung!** Be careful and considerate when scraping, and take care when writing and testing code to avoid unintended behavior. It is up to the programmer to respect the terms and conditions and `robots.txt` (which `urllib.robotparser` can read). Make sure to scrape websites legally.

`requests.get(url)` sends an HTTP GET request; the response has a `status_code` (200 is good), `ok`, `reason`, and the page source in `text`. Consecutive requests without pauses can strain a server and get the scraper blocked or blacklisted, so pause with `time.sleep()`: follow a `Request-rate` directive in `robots.txt` if there is one, otherwise half a second to a second is usually enough.

## Crawling through multiple pages

*Scraping* gathers data from a page; *crawling* navigates a program between pages. One approach follows links page by page: the lab's `scrape_books()` example walks the mystery category of books.toscrape.com (a practice bookstore site), collecting titles and following the relative "next" link (joined to the base URL), retrying each download until the expected tags appear.

## Problem 1: Mean price of fiction books

> **Problem 1.** Modify `scrape_books()` so that it gathers the price for each fiction book and returns the mean price, in £, of a fiction book.

In [ ]:
# Problem 1
def scrape_books(start_page = "index.html"):
    """ Crawl through http://books.toscrape.com and extract fiction data"""
    
    # Initialize variables, including a regex for finding the 'next' link.
    base_url="http://books.toscrape.com/catalogue/category/books/fiction_10/"
    price_total = []
    count_total = []
    page = base_url + start_page                # Complete page URL.
    next_page_finder = re.compile(r"next")      # We need this button.
    current = None

    for _ in range(4):
        total = 0
        count = 0
        
        while current == None:                   # Try downloading until it works.
            # Download the page source and PAUSE before continuing.  
            page_source = requests.get(page).text
            time.sleep(1)           # PAUSE before continuing.
            soup = BeautifulSoup(page_source, "html.parser")
            current = soup.find_all(class_="price_color")
            
        #Extract the price data.
        for price in current:
            total += float(price.string.strip("Â£"))
            count += 1

        price_total.append(total)
        count_total.append(count)
    

        if "page-4" not in page:
            # Find the URL for the page with the next data.
            new_page = soup.find(string=next_page_finder).parent["href"]    
            page = base_url + new_page      # New complete page URL.
            current = None
        else:
            # Calculate the price average
            average_price = sum(price_total)/sum(count_total)

    return average_price

The other approach first collects the links to all relevant pages, then scrapes each in turn. The lab's `bank_data()` example finds the Federal Reserve's large-bank release pages for 2004 and reads JPMorgan Chase's consolidated assets from each.

## Problem 2: Bank assets

> **Problem 2.** Modify `bank_data()` so that it extracts the total consolidated assets ("Consol Assets") for JPMorgan Chase, Bank of America, and Wells Fargo recorded each December from 2004 to the present. In a single figure, plot each bank's assets against time. Be careful to keep the data sorted by date.

In [ ]:
# Problem 2
def bank_data():
    """Crawl through the Federal Reserve site and extract bank data."""
    # Compile regular expressions for finding certain tags.
    link_finder = re.compile(r"^December 31, 20(1[0-8]|0[4-9])")
    chase_bank_finder = re.compile(r"^JPMORGAN CHASE BK")
    bank_of_america_finder = re.compile(r"^BANK OF AMER")
    wells_fargo_finder = re.compile(r"^WELLS FARGO BK")

    # Get the base page and find the URLs to all other relevant pages.
    base_url="https://www.federalreserve.gov/releases/lbr/"
    base_page_source = requests.get(base_url).text
    base_soup = BeautifulSoup(base_page_source, "html.parser")
    link_tags = base_soup.find_all(name='a', href=True, string=link_finder)
    pages = [base_url + tag.attrs["href"] for tag in link_tags]

    # Crawl through the individual pages and record the data.
    chase_assets = []
    boa_assets = []
    wf_assets = []
    for page in pages:
        time.sleep(1)               # PAUSE, then request the page.
        soup = BeautifulSoup(requests.get(page).text, "html.parser")
        print(page)
        # Find the tag corresponding to Chase Banks's consolidated assets.
        chase_temp_tag = soup.find(name="td", string=chase_bank_finder)
        boa_temp_tag = soup.find(name="td", string=bank_of_america_finder)
        wf_temp_tag = soup.find(name="td", string=wells_fargo_finder)
        for _ in range(10):
            chase_temp_tag = chase_temp_tag.next_sibling
            boa_temp_tag = boa_temp_tag.next_sibling
            wf_temp_tag = wf_temp_tag.next_sibling

        # Extract the data, removing commas.
        chase_assets.append(int(chase_temp_tag.string.replace(',', '')))
        boa_assets.append(int(boa_temp_tag.string.replace(',', '')))
        wf_assets.append(int(wf_temp_tag.string.replace(',', '')))

    # data to plot
    n_groups = 14
    means_frank = (90, 55, 40, 65)
    means_guido = (85, 62, 54, 20)
     
    # create plot
    fig, ax = plt.subplots()
    index = np.arange(n_groups)
    bar_width = 0.2
     
    rects1 = plt.bar(index, chase_assets, bar_width,
                     label='Chase Bank')
    rects2 = plt.bar(index + bar_width, boa_assets, bar_width,
                     label='Bank of America')
    rects2 = plt.bar(index + 2*bar_width, wf_assets, bar_width,
                     label='Wells Fargo')

    plt.xlabel('Year')
    plt.ylabel('Mil $')
    plt.title('Consolidated assets of each bank')
    plt.xticks(index + bar_width, [year for year in range(2017, 2003, -1)])
    plt.legend()
     
    plt.tight_layout()
    plt.show()

## Problem 3: NBA scoring leaders

> **Problem 3.** The Basketball Reference website at https://www.basketball-reference.com contains data on NBA athletes, including which player led different categories for each season. For the past ten seasons, identify which player had the most season points and find how many points they scored during that season. Return a list of triples consisting of the season, the player, and the points scored, `("season year", "player name", points scored)`.

In [ ]:
# Problem 3
def prob3():
    """The Basketball Reference website at https://www.basketball-reference.com
    contains data on NBA athletes, including which player led different categories 
    for each season. For the past ten seasons, identify which player had the most 
    season points and find how many points they scored during that season. Return 
    a list of triples consisting of the season, the player, and the points scored, 
    ("season year", "player name", points scored).
    """
    # Base url
    base_url = "https://www.basketball-reference.com"
    base_source = requests.get(base_url + "/leagues").text
    base_soup = BeautifulSoup(base_source, "html.parser")
    # Seasons we're interested in
    seasons = [str(year) for year in range(2018, 2008, -1)]
    most_points = []
    for season in seasons:
        year = season[-2:]
        season_str = str(int(season) - 1) + "-" + year
        row = base_soup.find_all(name='a', href=True, string=season_str)[1]
        season_tag = row.parent
        # Search to the right tag
        for i in range(5):
            season_tag = season_tag.next_sibling
        # Get the name and year
        name = ' '.join(season_tag.text.split()[:2])
        year = season_tag.text.split()[2].replace("(", "").replace(")", "")
        # Add to list
        most_points.append((season, name, int(year)))

    return most_points

## Asynchronously loaded content and user interaction

Plain requests fail when a page loads content with JavaScript after the initial source arrives, or when getting to the data needs clicks or typed input. **Selenium**, originally a web-app testing tool, drives a real browser instead: `webdriver.Chrome()` starts one, `get()` visits a page, `page_source` holds the current HTML (which can go to BeautifulSoup), and `close()` exits.

> **Note.** Selenium needs a driver executable for each browser (Chrome, Firefox, Safari, Opera, PhantomJS, ...); see the Selenium docs for installation and driver downloads.

Selenium also finds elements itself with `find_element_by_tag_name()`, `_by_name()`, `_by_class_name()`, `_by_id()`, `_by_link_text()` and `_by_partial_link_text()` (raising `NoSuchElementException` if nothing matches; the plural `find_elements_by_*()` return lists). Web elements can be clicked (`click()`), typed into (`send_keys()`, e.g. `Keys.RETURN` to press Enter), and cleared (`clear()`).

## Problem 4: arXiv search

> **Problem 4.** Use Selenium to enter the given search query into the search bar of https://arxiv.org and press Enter. The resulting page has up to 25 links to the PDFs of technical papers that match the query. Gather these URLs, then continue to the next page (if there are more results) and continue gathering links until obtaining at most 100 URLs. Return the list of URLs.

In [ ]:
# Problem 4
def prob4(search_query):
    """Use Selenium to enter the given search query into the search bar of
    https://arxiv.org and press Enter. The resulting page has up to 25 links
    to the PDFs of technical papers that match the query. Gather these URLs,
    then continue to the next page (if there are more results) and continue
    gathering links until obtaining at most 100 URLs. Return the list of URLs.

    Returns:
        (list): Up to 100 URLs that lead directly to PDFs on arXiv.
    """
    # Open browser through Selenium
    browser = webdriver.Chrome()
    urls = []
    try:
        # Open the webpage
        browser.get("https://arxiv.org")
        try:
            # Search the query
            search_bar = browser.find_element_by_name('query')
            search_bar.clear()
            search_bar.send_keys(search_query)
            search_bar.send_keys(Keys.RETURN)
            soup = BeautifulSoup(browser.page_source, "html.parser")
            # Find the urls of the pdfs
            url = soup.find_all(name='a', href=True, string='pdf')
            urls += url
            # Search through the next pages if it exists until we get 150 total
            try:
                while len(urls) < 150:
                    browser.find_element_by_class_name("pagination-next").send_keys(Keys.RETURN)
                    soup = BeautifulSoup(browser.page_source, "html.parser")
                    url = soup.find_all(name='a', href=True, string='pdf')
                    urls += url
            except NoSuchElementException:
                pass

        except NoSuchElementException:
            print("Could not find the search bar!")
            raise
    finally:
        # Close browser
        browser.close()

    return urls

> **Note.** Selenium is slower than `requests.get()` but safer, since it waits for each page to load; sites that are defensive about scrapers, like the arXiv, can still be read without offending their administrators.

## Problem 5: Project Euler

> **Problem 5.** For each of the (at least) 600 problems in the archive at https://projecteuler.net/archives, record the problem ID and the number of people who have solved it. Return a list of IDs, sorted from largest to smallest by the number of people who have solved them. That is, the first entry in the list should be the ID of the most solved problem, and the last entry in the list should be the ID of the least solved problem.

In [ ]:
# Problem 5
def prob5():
    """For each of the (at least) 600 problems in the archive at
    https://projecteuler.net/archives, record the problem ID and the number of
    people who have solved it. Return a list of IDs, sorted from largest to
    smallest by the number of people who have solved them. That is, the first
    entry in the list should be the ID of the most solved problem, and the
    last entry in the list should be the ID of the least solved problem.

    Returns:
        (list): problem IDs (as strings), from most solved to least solved.
    """    
    # Open broswer through Selenium
    browser = webdriver.Chrome()
    problems = []
    try:
        # Go to the webpage
        browser.get("https://projecteuler.net/archives")
        page = 1
        # Go through each of the pages
        while True:
            soup = BeautifulSoup(browser.page_source, "html.parser")
            # Regex for href
            re_href = re.compile(r"^problem=\d+$")
            # Find all the problems
            prob = soup.find_all(name='a', href=re_href)
            # Navigate to the tags with the id and num solved and add to list
            for p in prob:
                id_num = list(list(p.parents)[1].children)[0].text
                num_solved = list(list(p.parents)[1].children)[2].text
                problems.append([int(id_num), int(num_solved)])
            try:
                # Do this for all the other pages
                page += 1
                pg_str = "Go to page " + str(page)
                browser.find_element_by_xpath('//*[@title="{}"]'.format(pg_str)).send_keys(Keys.RETURN)
            except NoSuchElementException:
                print("{} doesn't exist".format(pg_str))
                break

    finally:
        # Close browser
        browser.close()

    problems.sort(key=lambda x: x[1], reverse=True)
    return [p[0] for p in problems]